Nome: Flávio Estevam Nogueira Andrade



1. Introdução

Esse projeto busca realizar a detecção de anomalias em transações financeiras, utilizando técnicas de clusterização não supervisionada para identificar transações fraudulentas ou anômalas em um conjunto de dados, por meio do algoritmo DBSCAN.

2. Coleta e Tratamento dos Dados


- A base de dados (https://www.kaggle.com/datasets/faizaniftikharjanjua/metaverse-financial-transactions-dataset) utilizada nesse projeto, com foco desta etapa sendo o pré-processamento dos dados para aplicação do algoritmo DBSCAN.

- Análise Inicial (df.info()): A primeira inspeção revelou que a base de dados não continha valores nulos, mas havia 8 colunas do tipo object que não podem ser processadas diretamente por algoritmos matemáticos.

- Remoção de Colunas: Colunas com alta cardinalidade, como timestamp e os endereços foram removidas, pois não agregariam ao modelo de clusterização por distância.

- Encoding: As colunas categóricas restantes, como transaction_type, location_region, etc. foram transformadas em colunas numéricas, de 0s e 1s, usando método One-Hot Encoding (pd.get_dummies).

- Padronização (StandardScaler): Devido ao algoritmo ser baseado em distância, os dados numéricos foram padronizados, garantindo que todas as variáveis tenham a mesma importância no cálculo da distância.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
import pprint

In [2]:
df = kagglehub.dataset_download("faizaniftikharjanjua/metaverse-financial-transactions-dataset")

print("Path to dataset files:", df)

100%|██████████| 5.17M/5.17M [00:00<00:00, 166MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/faizaniftikharjanjua/metaverse-financial-transactions-dataset/versions/1


In [3]:
folder_path = '/root/.cache/kagglehub/datasets/faizaniftikharjanjua/metaverse-financial-transactions-dataset/versions/1'

files_in_folder = os.listdir(folder_path)
print("Arquivos encontrados:", files_in_folder)

Arquivos encontrados: ['metaverse_transactions_dataset.csv']


In [4]:
csv_file_name = 'metaverse_transactions_dataset.csv'

full_file_path = os.path.join(folder_path, csv_file_name)

df = pd.read_csv(full_file_path)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78600 entries, 0 to 78599
Data columns (total 14 columns):
 #  Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0  timestamp          78600 non-null  object 
 1  hour_of_day        78600 non-null  int64  
 2  sending_address    78600 non-null  object 
 3  receiving_address  78600 non-null  object 
 4  amount             78600 non-null  float64
 5  transaction_type   78600 non-null  object 
 6  location_region    78600 non-null  object 
 7  ip_prefix         78600 non-null  float64
 8  login_frequency    78600 non-null  int64  
 9  session_duration   78600 non-null  int64  
 10 purchase_pattern   78600 non-null  object 
 11 age_group          78600 non-null  object 
 12 risk_score         78600 non-null  float64
 13 anomaly           78600 non-null  object 
dtypes: float64(3), int64(3), object(8)
memory usage: 8.4+ MB


In [6]:
coluna_objeto = df.select_dtypes(include=['object']).columns

print("Colunas do tipo 'object':")
print(coluna_objeto)

for coluna in coluna_objeto:
    num_unique = df[coluna].nunique()
    print(f"Coluna '{coluna}': {num_unique} valores únicos")

    if num_unique < 15:
        print(f"   Valores: {df[coluna].unique()}\n")
    else:
        print("\n")

Colunas do tipo 'object':
Index(['timestamp', 'sending_address', 'receiving_address', 'transaction_type',
       'location_region', 'purchase_pattern', 'age_group', 'anomaly'],
      dtype='object')
Coluna 'timestamp': 78513 valores únicos


Coluna 'sending_address': 1161 valores únicos


Coluna 'receiving_address': 1166 valores únicos


Coluna 'transaction_type': 5 valores únicos
   Valores: ['transfer' 'purchase' 'sale' 'phishing' 'scam']

Coluna 'location_region': 5 valores únicos
   Valores: ['Europe' 'South America' 'Asia' 'Africa' 'North America']

Coluna 'purchase_pattern': 3 valores únicos
   Valores: ['focused' 'high_value' 'random']

Coluna 'age_group': 3 valores únicos
   Valores: ['established' 'veteran' 'new']

Coluna 'anomaly': 3 valores únicos
   Valores: ['low_risk' 'moderate_risk' 'high_risk']



In [7]:
mapa_anomalia = {
    'baixo': 0,
    'médio': 1,
    'alto': 2
}

y = df['anomaly'].map(mapa_anomalia)

In [8]:
apagar_colunas = [
    'timestamp',
    'sending_address',
    'receiving_address',
    'anomaly'
]

remover = df.drop(columns = apagar_colunas)
remover

       hour_of_day      amount transaction_type location_region  ip_prefix  \
0               12  796.949206         transfer          Europe    192.000   
1               19    0.010000         purchase   South America    172.000   
2               16  778.197390         purchase            Asia    192.168   
3                9  300.838358         transfer   South America    172.000   
4               14  775.569344             sale          Africa    172.160   
...            ...         ...              ...             ...        ...   
78595           12  660.280373         transfer          Africa    172.000   
78596           16  310.273397         purchase          Africa    172.000   
78597           16  624.674332         purchase          Africa    192.000   
78598            4  401.391592         purchase            Asia    192.168   
78599           14  523.947956         transfer   North America    172.000   

       login_frequency session_duration purchase_pattern    age

In [9]:
encoded = pd.get_dummies(remover)
encoded.head()

   hour_of_day      amount  ip_prefix  login_frequency  session_duration  \
0           12  796.949206    192.000                3                48   
1           19    0.010000    172.000                5                61   
2           16  778.197390    192.168                3                74   
3            9  300.838358    172.000                8               111   
4           14  775.569344    172.160                6               100   

   risk_score transaction_type_phishing  transaction_type_purchase  \
0       18.75                     False                      False   
1       25.00                     False                       True   
2       31.25                     False                       True   
3       36.75                     False                      False   
4       62.50                     False                      False   

   transaction_type_sale  transaction_type_scam  ...  location_region_Asia  \
0                  False                  Fa

In [10]:
escala = StandardScaler()

x_escala = escala.fit_transform(encoded)
x_escala

array([[ 0.06738413,  1.19714679,  0.63924254, ...,  1.42100134,
        -0.70599378, -0.71160523],
       [ 1.07663271, -2.04380597,  0.35100702, ...,  1.42100134,
        -0.70599378, -0.71160523],
       [ 0.6440976 ,  1.12088784,  0.64166371, ...,  1.42100134,
        -0.70599378, -0.71160523],
       ...,
       [ 0.6440976 ,  0.49654789,  0.63924254, ..., -0.7037291 ,
         1.41644307, -0.71160523],
       [-1.08604282, -0.41148726,  0.64166371, ...,  1.42100134,
        -0.70599378, -0.71160523],
       [ 0.35574086,  0.08691887,  0.35100702, ...,  1.42100134,
        -0.70599378, -0.71160523]])

In [11]:
n_features = x_escala.shape[1]
min_samples = 2 * n_features

In [12]:
nbrs = NearestNeighbors(n_neighbors=min_samples).fit(x_escala)
distances, indices = nbrs.kneighbors(x_escala)

3. Aplicação e Explicação do Algoritmo (DBSCAN)

O DBSCAN é um algoritmo de clusterização baseado em densidade, escolhido por ser projetado especificamente para identificar outliers.


O DBSCAN depende de dois parâmetros:

- min_samples: foi o número mínimo de vizinhos que um ponto precisa ter para ser considerado "denso", nesse projeto resultou em 44.

- eps: foi a distância máxima para um ponto ser considerado "vizinho", sendo mostrado no gráfico ocorrendo em eps = 1.5.

In [13]:
k_distances = np.sort(distances[:, min_samples-1], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.title('Gráfico K-distance')
plt.xlabel('Ordenados por distância')
plt.ylabel(f'Distância do {min_samples}º Vizinho')
plt.grid(True)
plt.show()

<Figure size 1000x600 with 1 Axes>

In [14]:
dbscan = DBSCAN(eps = 1.5, min_samples = 44).fit_predict(x_escala)

unique_labels, counts = np.unique(dbscan, return_counts=True)
print("Contagem de clusters e anomalias:")

resultados = dict(zip(unique_labels, counts))

pprint.pprint(resultados)

Contagem de clusters e anomalias:
{np.int64(-1): np.int64(40),
 np.int64(0): np.int64(1306),
 np.int64(1): np.int64(2860),
 np.int64(2): np.int64(2868),
 np.int64(3): np.int64(1289),
 np.int64(4): np.int64(2907),
 np.int64(5): np.int64(1224),
 np.int64(6): np.int64(2975),
 np.int64(7): np.int64(1050),
 np.int64(8): np.int64(1192),
 np.int64(9): np.int64(2950),
 np.int64(10): np.int64(1024),
 np.int64(11): np.int64(2945),
 np.int64(12): np.int64(3005),
 np.int64(13): np.int64(749),
 np.int64(14): np.int64(714),
 np.int64(15): np.int64(2999),
 np.int64(16): np.int64(796),
 np.int64(17): np.int64(1321),
 np.int64(18): np.int64(1230),
 np.int64(19): np.int64(1027),
 np.int64(20): np.int64(726),
 np.int64(21): np.int64(750),
 np.int64(22): np.int64(310),
 np.int64(23): np.int64(2910),
 np.int64(24): np.int64(314),
 np.int64(25): np.int64(1043),
 np.int64(26): np.int64(723),
 np.int64(27): np.int64(1040),
 np.int64(28): np.int64(986),
 np.int64(29): np.int64(268),
 np.int64(30): np.int64(300

4. Apresentação e Análise dos Resultados

O DBSCAN classificou 40 transações como anomalias, restando agrupadas outros 90 clusters de comportamento "normal".

- Perfil de Risco:

Para entender o que torna essas 40 transações anômalas, foi feito cruzamento dos dados com o dataframe original para traçar um perfil.

O tipo mais comum nas anomalias foi 'phishing' .

O local mais comum foi 'Asia'.

In [15]:
filtrando_anomalias = (dbscan == -1)

df_anomalias = df[filtrando_anomalias]
print("O número de anomalias encontadas foi: ", {len(df_anomalias)})
print(df_anomalias.head())

O número de anomalias encontadas foi:  {40}
                 timestamp  hour_of_day  \
1466   2022-04-12 18:56:39           18   
4466   2022-01-10 08:44:21            8   
6250   2022-11-07 19:26:10           19   
8006   2022-10-03 06:17:32            6   
12476  2022-04-02 08:04:52            8   

                                   sending_address  \
1466   0x6751784c1de353d7b2fbe066fca3c9e11af191a5   
4466   0xc5f2ff2df2e759d8611325f0fdb684f1e243a1a1   
6250   0x87cd446adc9d04f59502281dd824caf141bdff9b   
8006   0x279a3f4d766593afa57ff1b8de71c9923ddaa02f   
12476  0xb10f08393f28f97566ae467a3eb41a2425b9eb5d   

                               receiving_address       amount  \
1466   0x068b2d763fa2b7bf7df1bbe747ce65665450be98  1461.780990   
4466   0x11f40ae67f6b648e8b4bbc2d1a04c665214f7d25  1485.989339   
6250   0x99464ee24511467e2d973b19a8f0d9a294a78b15     0.010000   
8006   0x2039d649d1cf690a7161e8cbff11c6f14e70ffa0     0.010000   
12476  0xfe2650f030f2c966775e11009cb015e8852ecf4

In [16]:
print("-- Tipo de transação --")

print("\nDistribuição nas anomalias encontradas:")
print(df_anomalias['transaction_type'].value_counts(normalize=True))

print("\nDistribuição GERAL (para comparação):")
print(df['transaction_type'].value_counts(normalize=True))

-- Tipo de transação --

Distribuição nas anomalias encontradas:
transaction_type
phishing    0.450
transfer    0.275
sale        0.125
purchase    0.100
scam        0.050
Name: proportion, dtype: float64

Distribuição GERAL (para comparação):
transaction_type
sale        0.318575
purchase    0.317303
transfer    0.281489
scam        0.050242
phishing    0.032392
Name: proportion, dtype: float64


In [17]:
print("\n-- Região --")
print("Nas anomalias:\n", df_anomalias['location_region'].value_counts(normalize=True))
print("\nGeral:\n", df['location_region'].value_counts(normalize=True))

print("\n-- Padrão de Compra --")
print("Nas anomalias:\n", df_anomalias['purchase_pattern'].value_counts(normalize=True))
print("\nGeral:\n", df['purchase_pattern'].value_counts(normalize=True))


-- Região --
Nas anomalias:
 location_region
Asia             0.300
North America    0.275
Europe           0.250
Africa           0.150
South America    0.025
Name: proportion, dtype: float64

Geral:
 location_region
North America    0.201527
Europe           0.201107
Asia             0.200140
South America    0.199351
Africa           0.197875
Name: proportion, dtype: float64

-- Padrão de Compra --
Nas anomalias:
 purchase_pattern
random        0.675
focused       0.175
high_value    0.150
Name: proportion, dtype: float64

Geral:
 purchase_pattern
high_value    0.336158
random        0.332634
focused       0.331209
Name: proportion, dtype: float64


In [18]:
colunas_numericas = ['amount', 'hour_of_day', 'login_frequency', 'session_duration', 'risk_score']

print("\n-- Valores Numéricos (Anomalias) --")
print(df_anomalias[colunas_numericas].describe())

print("\n-- Valores Numéricos (Geral) --")
print(df[colunas_numericas].describe())


-- Valores Numéricos (Anomalias) --
            amount  hour_of_day  login_frequency  session_duration  risk_score
count    40.000000    40.000000        40.000000         40.000000   40.000000
mean   1042.230052    11.575000         2.825000         48.300000   71.476562
std     511.478551     8.381566         2.170756         36.531827   31.307238
min        0.010000     0.000000         1.000000         20.000000   18.750000
25%     993.457088     4.000000         1.000000         26.000000   43.750000
50%    1268.279291     9.500000         2.000000         35.000000   92.187500
75%    1374.351923    20.250000         3.250000         52.250000  100.000000
max    1557.150905    23.000000         8.000000        158.000000  100.000000

-- Valores Numéricos (Geral) --
             amount   hour_of_day  login_frequency  session_duration  \
count  78600.000000  78600.000000     78600.000000      78600.000000   
mean     502.574903     11.532634         4.178702         69.684606   
st

In [19]:
print("-- Perfil Categórico da Anomalia --")
print(f"Tipo mais comum: {df_anomalias['transaction_type'].mode()[0]}")
print(f"Local mais comum: {df_anomalias['location_region'].mode()[0]}")

print(f"Padrão de compra mais comum: {df_anomalias['purchase_pattern'].mode()[0]}")
print(f"Grupo de idade mais comum: {df_anomalias['age_group'].mode()[0]}")

-- Perfil Categórico da Anomalia --
Tipo mais comum: phishing
Local mais comum: Asia
Padrão de compra mais comum: random
Grupo de idade mais comum: new


In [20]:
print("\n-- Perfil Numérico da Anomalia --")
colunas_numericas_perfil = ['amount', 'hour_of_day', 'login_frequency', 'session_duration', 'risk_score']
print(df_anomalias[colunas_numericas_perfil].describe())


-- Perfil Numérico da Anomalia --
            amount  hour_of_day  login_frequency  session_duration  risk_score
count    40.000000    40.000000        40.000000         40.000000   40.000000
mean   1042.230052    11.575000         2.825000         48.300000   71.476562
std     511.478551     8.381566         2.170756         36.531827   31.307238
min        0.010000     0.000000         1.000000         20.000000   18.750000
25%     993.457088     4.000000         1.000000         26.000000   43.750000
50%    1268.279291     9.500000         2.000000         35.000000   92.187500
75%    1374.351923    20.250000         3.250000         52.250000  100.000000
max    1557.150905    23.000000         8.000000        158.000000  100.000000


In [21]:
perfil_cruzado = df_anomalias.groupby(['transaction_type', 'location_region'])['amount'].describe()

print("\n--  Valor (Amount) por Tipo e Local --")
print(perfil_cruzado)

perfil_cruzado_horario = df_anomalias.groupby(['transaction_type', 'location_region'])['hour_of_day'].describe()

print("\n--  Horário por Tipo e Local --")
print(perfil_cruzado_horario)


--  Valor (Amount) por Tipo e Local --
                                  count         mean         std          min  \
transaction_type location_region                                                
phishing         Africa             1.0  1004.341525         NaN  1004.341525   
                 Asia               3.0  1098.927700   63.119619  1044.067320   
                 Europe             6.0   906.685828  479.177195    16.094534   
                 North America      8.0   553.560130  600.577433     0.010000   
purchase         Asia               3.0  1438.918857   91.151488  1338.512592   
                 Europe             1.0  1280.530863         NaN  1280.530863   
sale             Africa             2.0  1468.260038  125.710670  1379.369171   
                 Asia               2.0  1401.655113  119.266607  1317.320886   
                 Europe             1.0  1289.916140         NaN  1289.916140   
scam             Asia               2.0    13.567752   19.173557     

5. Conclusão

O DBSCAN provou ser eficaz para isolar transações suspeitas. o perfil da fraude identificada é caracterizada por transações do tipo 'phishing', originadas da 'Asia', com valores atipicamente altos e executadas em sessões de curtíssima duração.